## Notebook 4 — FAISS Candidate Retrieval

This is exactly the first stage used by:
* Netflix
* YouTube
* Spotify
* Perplexity Discover

### Install FAISS

In [1]:
!pip install faiss-cpu

### import required libraries

In [2]:
import pandas as pd
import pickle
import faiss
import numpy as np

### Load Files

In [3]:
items_df = pd.read_csv("data/item_df.csv")

Load item embeddings:

In [4]:
with open("data/item_embeddings.pkl", "rb") as f:
    item_embeddings = pickle.load(f)

Load user embeddings:

In [5]:
with open("data/user_embeddings.pkl", "rb") as f:
    user_embeddings = pickle.load(f)

### Convert to NumPy Array
FAISS requires float32.

In [6]:
item_embeddings = np.array(
    item_embeddings
).astype("float32")

In [7]:
item_embeddings.shape

(1000, 384)

### Create FAISS Index

In [8]:
dimension = item_embeddings.shape[1]

index = faiss.IndexFlatL2(
    dimension
)

### Add Item Embeddings

In [9]:
index.add(item_embeddings)

In [10]:
index.ntotal

1000

### Select User

In [11]:
user_vector = user_embeddings[1]

Convert:

In [12]:
user_vector = np.array(
    user_vector
).astype("float32")

### Reshape User Vector
FAISS expects:



In [13]:
user_vector = user_vector.reshape(1,-1)

In [14]:
user_vector.shape

(1, 384)

### Search Top 10 Candidates

In [15]:
D, I = index.search(
    user_vector,
    k=10
)

Where:
D = distances, 
I = indices

### View Candidate IDs

In [16]:
I

array([[658, 358, 357, 638, 758, 634, 369, 395, 767, 768]])

These are row indices.

### Convert Indices to Actual Items

In [18]:
recommended_items = items_df.iloc[
    I[0]
]

recommended_items

,item_id,title,category
658,659,Content 659,Business
358,359,Content 359,Machine Learning
357,358,Content 358,LLM
638,639,Content 639,Deep Learning
758,759,Content 759,Finance
634,635,Content 635,LLM
369,370,Content 370,Machine Learning
395,396,Content 396,Computer Vision
767,768,Content 768,Deep Learning
768,769,Content 769,Python


### Save Index

In [19]:
faiss.write_index(
    index,
    "data/faiss_index.bin"
)